# 02 — Data cleaning: canonical player-day activity

Read the two original ZIP archives directly and create a daily table with one row per
`UserID, Date`. Demographics retain their original values and row order, with only
datatype conversion and date parsing. No raw files are extracted, edited, or deleted.
No churn labels, predictive features, or machine learning are created.

**Cleaning policy:** sum all source records for each player-day, including exact
duplicates. Removing even an identical record would change the requested source totals;
there is no evidence here that it is an erroneous transaction. Zero values and negative
Winnings remain in the data. The accompanying
`data/raw/Codebook_for_Virtual_Casino_Gambling.pdf`, as confirmed in the project brief,
allows negative Winnings as accounting corrections; they are never removed, replaced,
or clipped. Aggregation may offset a negative correction against another record for
the same day. Diagnostic patterns alone do not establish a record's cause.

Outputs: `data/processed/demographics_clean.csv` and
`data/processed/daily_activity_clean.csv`. Re-running this notebook refreshes these
derived files only. The source archives are required locally.

## 1. Environment and read-only source loading

Locate the project from the repository root or `notebooks/`. Verify each archive and
require exactly one text member. Read tab-separated text without extraction. Hash the
raw inputs before and after execution to confirm they remain unchanged.

In [1]:
from pathlib import Path
import hashlib
import math
import zipfile
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 30)
pd.set_option('display.precision', 6)
project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / 'data/raw').is_dir()
                     and (p / 'notebooks/01_data_validation.ipynb').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Run from the project root or notebooks directory.')
RAW_DIR = project_root / 'data/raw'
PROCESSED_DIR = project_root / 'data/processed'
raw_paths = sorted(p for p in RAW_DIR.iterdir() if p.is_file())
raw_hashes = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in raw_paths}

def read_zip(name):
    with zipfile.ZipFile(RAW_DIR / name) as archive:
        assert archive.testzip() is None, f'Corrupt archive: {name}'
        members = [m.filename for m in archive.infolist()
                   if not m.is_dir() and Path(m.filename).suffix.lower() == '.txt']
        if len(members) != 1:
            raise ValueError(f'Expected one text member: {members}')
        with archive.open(members[0]) as source:
            return pd.read_csv(source, sep='\t', encoding='utf-8')

demographics_raw = read_zip('RawDataSet1_DemographicsCasinoTXT.zip')
daily_raw = read_zip('RawDataSet2_DailyAggregCasinoTXT.zip')
assert demographics_raw.columns.tolist() == [
    'UserID', 'Country', 'Language', 'Gender', 'RegDate', 'Fstcadate', 'Fstpdate']
assert daily_raw.columns.tolist() == ['UserID', 'Date', 'Stake', 'Winnings', 'Bets']
print('Demographics source shape:', demographics_raw.shape)
print('Daily activity source shape:', daily_raw.shape)

Demographics source shape: (4222, 7)
Daily activity source shape: (102165, 5)


## 2. Datatypes and all four date fields

Use the source's explicit month/day/year format (`%m/%d/%Y`) and fail on invalid
dates instead of silently dropping them. Identifiers, demographic category codes, and
Bets are integers; category codes retain their original meanings and values. Stake
and Winnings are numeric amounts. Reject missing or nonfinite values and fractional
integer fields rather than inventing replacements. No rounding, imputation, filtering,
or category remapping is performed.

In [2]:
demographics_clean = demographics_raw.copy(deep=True)
daily_activity = daily_raw.copy(deep=True)
date_rows = []
for name, frame, dates, integers in [
    ('Demographics', demographics_clean, ['RegDate', 'Fstcadate', 'Fstpdate'],
     ['UserID', 'Country', 'Language', 'Gender']),
    ('Daily activity', daily_activity, ['Date'], ['UserID', 'Bets']),
]:
    assert not frame.isna().any().any(), f'Missing source values in {name}'
    for column in dates:
        frame[column] = pd.to_datetime(frame[column], format='%m/%d/%Y', errors='raise')
        date_rows.append({'dataset': name, 'field': column,
                          'start': frame[column].min(), 'end': frame[column].max()})
    for column in integers:
        numeric = pd.to_numeric(frame[column], errors='raise')
        assert np.isfinite(numeric).all() and numeric.mod(1).eq(0).all(), column
        frame[column] = numeric.astype('int64')
for column in ['Stake', 'Winnings']:
    daily_activity[column] = pd.to_numeric(daily_activity[column], errors='raise').astype('float64')
assert np.isfinite(daily_activity[['Stake', 'Winnings', 'Bets']]).all().all()
assert daily_activity['Stake'].ge(0).all()
assert daily_activity['Bets'].ge(0).all()
assert demographics_clean['UserID'].is_unique
assert set(demographics_clean['UserID']) == set(daily_activity['UserID'])
demographics_baseline = demographics_clean.copy(deep=True)
display(pd.DataFrame(date_rows))
display(demographics_clean.dtypes.rename('dtype').to_frame())
display(daily_activity.dtypes.rename('dtype').to_frame())

,dataset,field,start,end
0,Demographics,RegDate,2005-02-01,2005-02-28
1,Demographics,Fstcadate,2005-02-01,2007-01-30
2,Demographics,Fstpdate,2005-02-01,2007-01-25
3,Daily activity,Date,2005-02-01,2007-02-28


,dtype
UserID,int64
Country,int64
Language,int64
Gender,int64
RegDate,datetime64[us]
Fstcadate,datetime64[us]
Fstpdate,datetime64[us]


,dtype
UserID,int64
Date,datetime64[us]
Stake,float64
Winnings,float64
Bets,int64


## 3. Exact duplicates versus non-identical repeated player-days

An **exact duplicate extra row** is a full-row match beyond the first occurrence;
**rows involved** includes every occurrence. Repeated keys are counted separately.
After deduplicating a diagnostic copy only, keys with more than one distinct full row
are non-identical repeated keys. Their row count includes all original records at
those keys. These definitions also handle keys containing both exact duplicates and
different records. The table used for aggregation remains intact.

Compare each metric's number of distinct values, minimum, maximum, and sum, and show
complete source groups for representative keys. This documents differences without
assuming why multiple records exist.

In [3]:
KEY = ['UserID', 'Date']
METRICS = ['Stake', 'Winnings', 'Bets']
exact_extra = int(daily_activity.duplicated().sum())
exact_mask = daily_activity.duplicated(keep=False)
key_sizes = daily_activity.groupby(KEY).size()
distinct_sizes = daily_activity.drop_duplicates().groupby(KEY).size()
repeated_sizes = key_sizes[key_sizes.gt(1)]
nonidentical_keys = distinct_sizes[distinct_sizes.gt(1)].index
identical_only_keys = repeated_sizes.index.difference(nonidentical_keys)
duplicate_summary = pd.Series({
    'exact_duplicate_extra_rows': exact_extra,
    'rows_involved_in_exact_duplicates': int(exact_mask.sum()),
    'distinct_exact_duplicate_patterns': len(daily_activity.loc[exact_mask].drop_duplicates()),
    'repeated_player_date_keys': len(repeated_sizes),
    'rows_in_repeated_keys': int(repeated_sizes.sum()),
    'extra_rows_beyond_one_per_key': int(daily_activity.duplicated(KEY).sum()),
    'nonidentical_repeated_keys': len(nonidentical_keys),
    'original_rows_in_nonidentical_repeated_keys': int(key_sizes.loc[nonidentical_keys].sum()),
    'extra_distinct_rows_beyond_one_per_key': int((distinct_sizes - 1).sum()),
    'identical_only_repeated_keys': len(identical_only_keys),
}, name='count')
display(duplicate_summary.to_frame())
print('Exact full-row duplicate examples (all occurrences):')
display(daily_activity.loc[exact_mask].sort_values(KEY).head(12))
indexed_activity = daily_activity.set_index(KEY)
print('Non-identical repeated player-date examples (complete groups):')
display(indexed_activity.loc[nonidentical_keys[:5]].reset_index())
metric_comparison = daily_activity.groupby(KEY)[METRICS].agg(['nunique', 'min', 'max', 'sum'])
print('Metric comparisons for those groups:')
display(metric_comparison.loc[nonidentical_keys[:5]])
print('Repeated keys whose records differ in each metric (counts can overlap):')
display(pd.Series({m: int(metric_comparison.loc[repeated_sizes.index, (m, 'nunique')].gt(1).sum())
                   for m in METRICS}, name='key_count').to_frame())

,count
exact_duplicate_extra_rows,1
rows_involved_in_exact_duplicates,2
distinct_exact_duplicate_patterns,1
repeated_player_date_keys,3160
rows_in_repeated_keys,6334
extra_rows_beyond_one_per_key,3174
nonidentical_repeated_keys,3159
original_rows_in_nonidentical_repeated_keys,6332
extra_distinct_rows_beyond_one_per_key,3173
identical_only_repeated_keys,1


Exact full-row duplicate examples (all occurrences):


,UserID,Date,Stake,Winnings,Bets
11594,1378228,2005-12-22,1.0,2.0,1
11595,1378228,2005-12-22,1.0,2.0,1


Non-identical repeated player-date examples (complete groups):


,UserID,Date,Stake,Winnings,Bets
0,1324369,2005-12-20,23.0,14.5,8
1,1324369,2005-12-20,25.0,15.0,25
2,1324369,2006-01-10,197.0,156.5,19
3,1324369,2006-01-10,179.0,179.5,99
4,1324369,2006-01-21,423.0,380.0,14
5,1324369,2006-01-21,1208.0,1158.0,279
6,1324369,2006-07-05,11.0,12.5,11
7,1324369,2006-07-05,167.0,134.5,17
8,1324369,2007-01-06,5.0,2.0,1
9,1324369,2007-01-06,61.5,46.5,60


Metric comparisons for those groups:


Stake                        Winnings                 \
                   nunique    min     max     sum  nunique    min     max   
UserID  Date                                                                
1324369 2005-12-20       2   23.0    25.0    48.0        2   14.5    15.0   
        2006-01-10       2  179.0   197.0   376.0        2  156.5   179.5   
        2006-01-21       2  423.0  1208.0  1631.0        2  380.0  1158.0   
        2006-07-05       2   11.0   167.0   178.0        2   12.5   134.5   
        2007-01-06       2    5.0    61.5    66.5        2    2.0    46.5   

                              Bets                
                       sum nunique min  max  sum  
UserID  Date                                      
1324369 2005-12-20    29.5       2   8   25   33  
        2006-01-10   336.0       2  19   99  118  
        2006-01-21  1538.0       2  14  279  293  
        2006-07-05   147.0       2  11   17   28  
        2007-01-06    48.5       2   1   60   61

Repeated keys whose records differ in each metric (counts can overlap):


,key_count
Stake,3149
Winnings,3139
Bets,3112


## 4. Zero amounts, zero Bets, and negative Winnings in source records

Count the requested zero-value patterns independently: zero Stake with positive
Winnings can overlap zero Stake with positive Bets or zero Bets. Show examples for
each pattern. Zero Bets are split into nonzero Stake, nonzero Winnings, either nonzero,
and both zero. Retain all such records: activity or accounting entries can have unusual
combinations, and this audit alone does not justify deleting them.

Retain every negative Winnings value as an accounting correction allowed by the
codebook. Display negative examples explicitly, including corrections with no Stake
or Bets, without reclassifying or clipping them.

In [4]:
def zero_audit(frame, label, show_examples=True):
    masks = {
        'Stake = 0 (all)': frame['Stake'].eq(0),
        'Stake = 0 and Winnings > 0': frame['Stake'].eq(0) & frame['Winnings'].gt(0),
        'Stake = 0 and Bets > 0': frame['Stake'].eq(0) & frame['Bets'].gt(0),
        'Stake = 0 and Bets = 0': frame['Stake'].eq(0) & frame['Bets'].eq(0),
        'Bets = 0 (all)': frame['Bets'].eq(0),
        'Bets = 0 and Stake != 0': frame['Bets'].eq(0) & frame['Stake'].ne(0),
        'Bets = 0 and Winnings != 0': frame['Bets'].eq(0) & frame['Winnings'].ne(0),
        'Bets = 0 and either amount != 0': frame['Bets'].eq(0) &
            (frame['Stake'].ne(0) | frame['Winnings'].ne(0)),
        'Bets = 0 and both amounts = 0': frame['Bets'].eq(0) &
            frame['Stake'].eq(0) & frame['Winnings'].eq(0),
        'Winnings < 0': frame['Winnings'].lt(0),
    }
    counts = pd.Series({name: int(mask.sum()) for name, mask in masks.items()}, name=label)
    display(counts.to_frame())
    if show_examples:
        for name, mask in masks.items():
            print(f'{label}: {name}; examples (up to five)')
            display(frame.loc[mask].head(5))
    return counts

source_zero_counts = zero_audit(daily_activity, 'Source rows')

,Source rows
Stake = 0 (all),5044
Stake = 0 and Winnings > 0,109
Stake = 0 and Bets > 0,4995
Stake = 0 and Bets = 0,49
Bets = 0 (all),49
Bets = 0 and Stake != 0,0
Bets = 0 and Winnings != 0,49
Bets = 0 and either amount != 0,49
Bets = 0 and both amounts = 0,0
Winnings < 0,72


Source rows: Stake = 0 (all); examples (up to five)


,UserID,Date,Stake,Winnings,Bets
3,1333544,2007-02-25,0.0,-91.0,0
4,1345514,2006-12-28,0.0,-50.0,0
8,1335498,2005-03-05,0.0,-25.0,0
14,1350206,2006-08-01,0.0,-15.0,0
16,1364109,2005-03-02,0.0,-13.0,0


Source rows: Stake = 0 and Winnings > 0; examples (up to five)


,UserID,Date,Stake,Winnings,Bets
10311,1353561,2006-07-01,0.0,1.0,0
11003,1326008,2006-01-04,0.0,2.0,0
11058,1330825,2006-04-21,0.0,2.0,0
11370,1353561,2005-10-24,0.0,2.0,0
11485,1365491,2006-09-27,0.0,2.0,0


...

,UserID,Date,Stake,Winnings,Bets
3,1333544,2007-02-25,0.0,-91.0,0
4,1345514,2006-12-28,0.0,-50.0,0
8,1335498,2005-03-05,0.0,-25.0,0
14,1350206,2006-08-01,0.0,-15.0,0
16,1364109,2005-03-02,0.0,-13.0,0


Source rows: Bets = 0 and both amounts = 0; examples (up to five)


,UserID,Date,Stake,Winnings,Bets


Source rows: Winnings < 0; examples (up to five)


,UserID,Date,Stake,Winnings,Bets
0,1363150,2006-07-05,485.0,-150.0,4
1,1383941,2006-08-21,16.5,-121.0,8
2,1391558,2006-09-08,4.0,-100.0,2
3,1333544,2007-02-25,0.0,-91.0,0
4,1345514,2006-12-28,0.0,-50.0,0


## 5. Canonical daily aggregation and conservation checks

Group all source rows by `UserID, Date` and sum Stake, Winnings, and Bets. Sort the
result by player and date for reproducible exports. Do not remove the exact duplicate:
this explicit policy preserves the totals of the supplied source. It can be reconsidered
only with evidence and a separately agreed policy.

Use `math.fsum` to compare monetary totals with **absolute tolerance 0.000001 and no
relative tolerance**, because floating-point addition can differ by summation order.
No amounts are rounded to make the checks pass. Bets are integers and must match
exactly. Require the entire player ID set and both date endpoints to match, no missing
values, and no duplicate player-date keys. Demographics must match the typed, parsed
baseline exactly, including row order.

In [5]:
daily_activity_clean = daily_activity.groupby(KEY, as_index=False, sort=True)[METRICS].sum()
conservation_rows = []
for metric in METRICS:
    before = int(daily_activity[metric].sum()) if metric == 'Bets' else math.fsum(daily_activity[metric])
    after = int(daily_activity_clean[metric].sum()) if metric == 'Bets' else math.fsum(daily_activity_clean[metric])
    matches = before == after if metric == 'Bets' else math.isclose(before, after, rel_tol=0, abs_tol=1e-6)
    conservation_rows.append({'metric': metric, 'before': before, 'after': after,
                              'difference': after - before, 'matches': matches})
conservation = pd.DataFrame(conservation_rows).set_index('metric')
display(conservation)
assert conservation['matches'].all(), 'Aggregation changed a metric total'
players_before = daily_activity['UserID'].nunique()
players_after = daily_activity_clean['UserID'].nunique()
range_before = (daily_activity['Date'].min(), daily_activity['Date'].max())
range_after = (daily_activity_clean['Date'].min(), daily_activity_clean['Date'].max())
assert set(daily_activity['UserID']) == set(daily_activity_clean['UserID'])
assert range_before == range_after
assert not daily_activity_clean.duplicated(KEY).any()
assert not daily_activity_clean.isna().any().any()
assert np.isfinite(daily_activity_clean[METRICS]).all().all()
assert len(daily_activity_clean) == len(key_sizes)
pd.testing.assert_frame_equal(demographics_clean, demographics_baseline)
print('All aggregation and demographics preservation checks passed.')

,before,after,difference,matches
metric,,,,
Stake,1.147184e+08,1.147184e+08,0.0,True
Winnings,1.111732e+08,1.111732e+08,0.0,True
Bets,1.484053e+07,1.484053e+07,0.0,True


All aggregation and demographics preservation checks passed.


## 6. Zero-value patterns after aggregation

Repeat the audit on canonical player-days. Counts may differ from source row counts
because multiple records become one day and amounts combine. A negative Winnings
record still contributes its signed value even when the resulting daily total is
positive. Neither the source nor canonical zero-value patterns trigger deletion.

In [6]:
clean_zero_counts = zero_audit(daily_activity_clean, 'Canonical days')
print('Source versus canonical pattern counts (overlapping categories):')
display(pd.concat([source_zero_counts, clean_zero_counts], axis=1))

,Canonical days
Stake = 0 (all),4896
Stake = 0 and Winnings > 0,77
Stake = 0 and Bets > 0,4854
Stake = 0 and Bets = 0,42
Bets = 0 (all),42
Bets = 0 and Stake != 0,0
Bets = 0 and Winnings != 0,42
Bets = 0 and either amount != 0,42
Bets = 0 and both amounts = 0,0
Winnings < 0,69


Canonical days: Stake = 0 (all); examples (up to five)


,UserID,Date,Stake,Winnings,Bets
2,1324364,2006-09-16,0.0,0.0,14
42,1324369,2006-03-11,0.0,0.0,1
44,1324369,2006-04-30,0.0,0.0,4
45,1324369,2006-06-05,0.0,0.0,7
244,1324582,2006-06-06,0.0,0.0,1


Canonical days: Stake = 0 and Winnings > 0; examples (up to five)


,UserID,Date,Stake,Winnings,Bets
4349,1327714,2005-04-22,0.0,7.0,1
7261,1330283,2005-04-22,0.0,5.0,1
8477,1330825,2006-04-21,0.0,2.0,0
8641,1331020,2005-05-04,0.0,5.0,1
10092,1331917,2005-03-03,0.0,25.0,1


...

,UserID,Date,Stake,Winnings,Bets


Canonical days: Winnings < 0; examples (up to five)


,UserID,Date,Stake,Winnings,Bets
203,1324533,2005-02-09,3.0,-1.00,3
1332,1325682,2005-10-19,3.0,-39.00,2
1544,1325824,2005-06-05,0.0,-5.00,0
1662,1325854,2005-12-29,7.0,-2.00,4
1667,1325854,2006-07-11,4.0,-1.25,4


Source versus canonical pattern counts (overlapping categories):


,Source rows,Canonical days
Stake = 0 (all),5044,4896
Stake = 0 and Winnings > 0,109,77
Stake = 0 and Bets > 0,4995,4854
Stake = 0 and Bets = 0,49,42
Bets = 0 (all),49,42
Bets = 0 and Stake != 0,0,0
Bets = 0 and Winnings != 0,49,42
Bets = 0 and either amount != 0,49,42
Bets = 0 and both amounts = 0,0,0
Winnings < 0,72,69


## 7. Print the final checks before saving

Report both shapes, unique players, duplicate keys, every date field's range, and all
three totals before writing anything. The monetary match flags use the stated
floating-point tolerance; Bets match exactly.

In [7]:
final_summary = pd.Series({
    'demographics_clean_shape': str(demographics_clean.shape),
    'daily_activity_clean_shape': str(daily_activity_clean.shape),
    'exact_duplicate_extra_source_rows': exact_extra,
    'daily_rows_before': len(daily_activity),
    'daily_rows_after': len(daily_activity_clean),
    'demographics_unique_players': demographics_clean['UserID'].nunique(),
    'daily_unique_players_before': players_before,
    'daily_unique_players_after': players_after,
    'duplicate_UserID_Date_after': int(daily_activity_clean.duplicated(KEY).sum()),
    'daily_date_start_before': range_before[0],
    'daily_date_end_before': range_before[1],
    'daily_date_start_after': range_after[0],
    'daily_date_end_after': range_after[1],
    'total_Stake': math.fsum(daily_activity_clean['Stake']),
    'total_Winnings': math.fsum(daily_activity_clean['Winnings']),
    'total_Bets': int(daily_activity_clean['Bets'].sum()),
    'Stake_totals_match': bool(conservation.loc['Stake', 'matches']),
    'Winnings_totals_match': bool(conservation.loc['Winnings', 'matches']),
    'Bets_totals_match': bool(conservation.loc['Bets', 'matches']),
}, name='result')
display(final_summary.to_frame())
for label, frame, columns in [
    ('Demographics', demographics_clean, ['RegDate', 'Fstcadate', 'Fstpdate']),
    ('Canonical daily activity', daily_activity_clean, ['Date']),
]:
    print(label, 'shape:', frame.shape, '; unique players:', frame['UserID'].nunique())
    for column in columns:
        print(column, frame[column].min().date(), 'to', frame[column].max().date())
print('Duplicate UserID-Date count:', daily_activity_clean.duplicated(KEY).sum())
print('Total Stake:', format(math.fsum(daily_activity_clean['Stake']), '.10f'))
print('Total Winnings:', format(math.fsum(daily_activity_clean['Winnings']), '.10f'))
print('Total Bets:', int(daily_activity_clean['Bets'].sum()))

,result
demographics_clean_shape,"(4222, 7)"
daily_activity_clean_shape,"(98991, 5)"
exact_duplicate_extra_source_rows,1
daily_rows_before,102165
daily_rows_after,98991
demographics_unique_players,4222
daily_unique_players_before,4222
daily_unique_players_after,4222
duplicate_UserID_Date_after,0
daily_date_start_before,2005-02-01 00:00:00


Demographics shape: (4222, 7) ; unique players: 4222
RegDate 2005-02-01 to 2005-02-28
Fstcadate 2005-02-01 to 2007-01-30
Fstpdate 2005-02-01 to 2007-01-25
Canonical daily activity shape: (98991, 5) ; unique players: 4222
Date 2005-02-01 to 2007-02-28
Duplicate UserID-Date count: 0
Total Stake: 114718425.3357773870


Total Winnings: 111173169.1709691137


Total Bets: 14840532


## 8. Save and verify the cleaned CSV files

Write only the two processed outputs, without a CSV index and with ISO `YYYY-MM-DD`
dates. CSV does not store dtype metadata; explicitly restore integer and date types
when reloading. Read both exports back and verify their values against the in-memory
tables (allowing floating-point serialization tolerance for amounts). Confirm the raw
file inventory and SHA-256 hashes are unchanged. No version-control commands are run.

In [8]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
demographics_path = PROCESSED_DIR / 'demographics_clean.csv'
daily_path = PROCESSED_DIR / 'daily_activity_clean.csv'
demographics_clean.to_csv(demographics_path, index=False, date_format='%Y-%m-%d')
daily_activity_clean.to_csv(daily_path, index=False, date_format='%Y-%m-%d')

demographics_reload = pd.read_csv(demographics_path,
    dtype={c: 'int64' for c in ['UserID', 'Country', 'Language', 'Gender']})
daily_reload = pd.read_csv(daily_path, dtype={'UserID': 'int64', 'Bets': 'int64'},
                           float_precision='round_trip')
for column in ['RegDate', 'Fstcadate', 'Fstpdate']:
    demographics_reload[column] = pd.to_datetime(demographics_reload[column], format='%Y-%m-%d', errors='raise')
daily_reload['Date'] = pd.to_datetime(daily_reload['Date'], format='%Y-%m-%d', errors='raise')
pd.testing.assert_frame_equal(demographics_reload, demographics_clean)
pd.testing.assert_frame_equal(daily_reload, daily_activity_clean, check_exact=False, rtol=0, atol=1e-10)
raw_paths_after = sorted(p for p in RAW_DIR.iterdir() if p.is_file())
assert raw_paths_after == raw_paths, 'Raw file inventory changed'
assert {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in raw_paths_after} == raw_hashes
print('Saved:', demographics_path.relative_to(project_root))
print('Saved:', daily_path.relative_to(project_root))
print('CSV reload checks passed; all raw files unchanged. Cleaning complete.')

Saved: data\processed\demographics_clean.csv
Saved: data\processed\daily_activity_clean.csv
CSV reload checks passed; all raw files unchanged. Cleaning complete.
